# NYISO DART Strategy — Interactive Sandbox

The 22 logistic regression models are already trained on 2015–2019 data and live on disk.  
This notebook lets you **test any set of inputs** without re-training anything.

## Two ways to use this

| Mode | What you do | When to use |
|---|---|---|
| **Dataset lookup** | Specify a date + hour from our historical data | Check what the model actually predicted on any day in 2015–2026 |
| **Manual input** | Fill in a Python dictionary of 50 feature values | Hypothetical scenarios, sensitivity analysis, future data |

Run the **Setup** section once, then use whichever mode you want.

---
**Model trained on**: 2015–2019  
**Thresholds tuned on**: 2020–2021  
**Nothing is re-fitted here** — all weights are loaded from disk.

---
## Setup — run this cell once before anything else

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import holidays

# Project paths
PROJECT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT))
DATA   = PROJECT / 'data'
MODELS = PROJECT / 'models'

# ── Feature column order (must match the order in X_naive.parquet exactly) ────
ZONES = ['CAPITL','CENTRL','DUNWOD','GENESE','HUDVL',
         'LONGIL','MHKVL','MILLWD','NORTH','NYC','WEST']
ZONE_FEATS = ['da_load_forecast','dart_lag24','dart_lag48','lfe_lag24']
CAL_FEATS  = ['hour_of_day','month_of_year','is_winter','is_summer','is_weekend','is_holiday']

# Exact column order the models expect
FEATURE_COLS = [f'{z}_{f}' for f in ZONE_FEATS for z in ZONES] + CAL_FEATS
assert len(FEATURE_COLS) == 50, f'Expected 50 features, got {len(FEATURE_COLS)}'

# Paper constants
GAMMA_POS = 5.0    # $/MWh — positive DART spike threshold (DEC label)
GAMMA_NEG = 30.0   # $/MWh — negative DART spike threshold (INC label)

# ── Load all 22 models from disk ──────────────────────────────────────────────
PIPES = {}
for zone in ZONES:
    for side in ('pos', 'neg'):
        PIPES[f'{zone}_{side}'] = joblib.load(MODELS / 'naive' / f'{zone}_{side}.joblib')

# ── Load validation-tuned thresholds ──────────────────────────────────────────
THR = pd.read_parquet(MODELS / 'thresholds_naive.parquet')
THR = THR.set_index(['zone','side'])

# ── US holiday calendar ───────────────────────────────────────────────────────
US_HOLIDAYS = holidays.country_holidays('US', subdiv='NY')

print(f'Loaded {len(PIPES)} models')
print(f'Eligible (zone, side) pairs: {THR["eligible"].sum()}/22')
print(f'Feature columns: {len(FEATURE_COLS)}')
print()
print('Ready. Use MODE 1 or MODE 2 below.')

Loaded 22 models
Eligible (zone, side) pairs: 13/22
Feature columns: 50

Ready. Use MODE 1 or MODE 2 below.


---
## Core prediction function

Everything passes through `predict()`. It accepts a dictionary of 50 feature values  
and returns a table of probabilities and trade decisions for all 22 (zone, side) pairs.

In [ ]:
def predict(features: dict, verbose: bool = True) -> pd.DataFrame:
    """
    Run all 22 logistic regression models on a single feature vector.

    Parameters
    ----------
    features : dict
        50-key dictionary. Keys must be exactly the strings in FEATURE_COLS.
        Missing keys raise KeyError. Extra keys are ignored.

    Returns
    -------
    pd.DataFrame with columns:
        zone, side, probability, threshold, eligible, trade_fires,
        trade_type, expected_payoff_direction
    """
    # Build the 50-element array in the exact column order the scaler expects
    try:
        x = np.array([features[c] for c in FEATURE_COLS], dtype=float)
    except KeyError as e:
        raise KeyError(f'Missing feature: {e}. Required keys: {FEATURE_COLS}') from e

    if np.any(np.isnan(x)):
        nan_cols = [FEATURE_COLS[i] for i in np.where(np.isnan(x))[0]]
        print(f'Warning: NaN values in features: {nan_cols}')
        print('Predictions will be NaN for affected models.')

    rows = []
    for zone in ZONES:
        for side in ('pos', 'neg'):
            label = f'{zone}_{side}'
            pipe  = PIPES[label]

            # Scale + predict (Pipeline handles both steps)
            p = float(pipe.predict_proba(x.reshape(1, -1))[0, 1])

            thr_row  = THR.loc[(zone, side)]
            tau      = float(thr_row['best_tau']) if pd.notna(thr_row['best_tau']) else None
            eligible = bool(thr_row['eligible'])
            fires    = eligible and (tau is not None) and (p >= tau)

            rows.append({
                'zone':       zone,
                'side':       side,
                'probability':p,
                'threshold':  tau,
                'eligible':   eligible,
                'trade_fires':fires,
                'trade_type': ('DEC (sell DA, buy RT)' if side == 'pos' else
                               'INC (buy DA, sell RT)') if fires else '—',
                'profit_when':('DART > 0' if side == 'pos' else 'DART < 0') if fires else '—',
            })

    result = pd.DataFrame(rows)

    if verbose:
        fired = result[result['trade_fires']]
        print(f'Trades that would fire: {len(fired)} of {len(result)} (zone, side) pairs')
        print()
        print('All 22 predictions:')
        display_df = result[['zone','side','probability','threshold','eligible','trade_fires','trade_type']].copy()
        display_df['probability'] = display_df['probability'].map(lambda x: f'{x:.1%}')
        display_df['threshold']   = display_df['threshold'].map(
            lambda x: f'{x:.2f}' if pd.notna(x) else '—')
        print(display_df.to_string(index=False))

        if len(fired):
            print()
            print('Trades that FIRE:')
            for _, r in fired.iterrows():
                print(f'  {r.zone} {r.side.upper()}  '
                      f'p={float(result.loc[result.zone==r.zone][result.side==r.side]["probability"].iloc[0]):.1%}  '
                      f'tau={r.threshold:.2f}  {r.trade_type}')

    return result


def calendar_features(dt: pd.Timestamp) -> dict:
    """Compute the 6 calendar features for any tz-aware timestamp."""
    return {
        'hour_of_day':   dt.hour,
        'month_of_year': dt.month,
        'is_winter':     int(dt.month in (12, 1, 2)),
        'is_summer':     int(dt.month in (6, 7, 8)),
        'is_weekend':    int(dt.weekday() >= 5),
        'is_holiday':    int(dt.date() in US_HOLIDAYS),
    }

print('predict() and calendar_features() defined.')

---
## MODE 1 — Look up any date and hour from the dataset

This pulls the exact feature vector from our historical data (2015–2025, plus 2026 YTD if available).  
**Just change `TARGET_DATE` and `TARGET_HOUR` and re-run.**

In [ ]:
# ── USER INPUT ─────────────────────────────────────────────────────────────────
TARGET_DATE = '2025-04-10'   # any date between 2015-01-01 and 2026-05-15
TARGET_HOUR = 12              # 0–23 (hour beginning, Eastern time)
# ──────────────────────────────────────────────────────────────────────────────

# Load whichever panel has this date
ts = pd.Timestamp(f'{TARGET_DATE} {TARGET_HOUR:02d}:00:00', tz='America/New_York')

panel_path = DATA / 'processed' / 'panel.parquet'
live_path  = DATA / 'processed' / 'panel_live_2026.parquet'

# Try main panel first, fall back to 2026 panel
panel = pd.read_parquet(panel_path)
row_check = panel[panel['interval_start_local'] == ts]
if row_check.empty and live_path.exists():
    panel = pd.read_parquet(live_path)
    row_check = panel[panel['interval_start_local'] == ts]

if row_check.empty:
    print(f'No data found for {ts}.')
    print('Check that the date is within 2015-01-01 to 2026-05-15.')
else:
    # Build the feature dict from panel data
    dart_wide = panel.pivot(index='interval_start_local', columns='zone',
                            values='dart')[ZONES]
    lf_wide   = panel.pivot(index='interval_start_local', columns='zone',
                            values='da_load_forecast')[ZONES]
    lfe_wide  = panel.pivot(index='interval_start_local', columns='zone',
                            values='load_forecast_error')[ZONES]
    actual_wide = panel.pivot(index='interval_start_local', columns='zone',
                              values='actual_load')[ZONES]

    lag24 = ts - pd.Timedelta(hours=24)
    lag48 = ts - pd.Timedelta(hours=48)

    features = {}
    for z in ZONES:
        features[f'{z}_da_load_forecast'] = float(lf_wide.loc[ts, z]) if ts in lf_wide.index else float('nan')
        features[f'{z}_dart_lag24']  = float(dart_wide.loc[lag24, z]) if lag24 in dart_wide.index else float('nan')
        features[f'{z}_dart_lag48']  = float(dart_wide.loc[lag48, z]) if lag48 in dart_wide.index else float('nan')
        features[f'{z}_lfe_lag24']   = float(lfe_wide.loc[lag24, z])  if lag24 in lfe_wide.index else float('nan')
    features.update(calendar_features(ts))

    # Show what the raw panel data looks like for this hour
    hour_panel = row_check[['zone','da_lmp','rt_lmp','dart',
                            'da_load_forecast','actual_load','load_forecast_error']]
    print(f'Raw panel data for {ts.strftime("%H:%M %A %B %d, %Y")}:')
    print(hour_panel.to_string(index=False))
    print()

    # Run the model
    print('=' * 60)
    print(f'MODEL PREDICTIONS for {ts.strftime("%H:%M %b %d, %Y")}')
    print('=' * 60)
    results = predict(features)

---
## MODE 2 — Enter data manually

Fill in the dictionary below with any values you want.  
Each key is exactly `{ZONE}_{feature}` for the 44 zone features, plus 6 calendar features.

The values below are a **realistic example of a summer peak hour**  
(high load forecasts, elevated lagged DARTs from the prior day).

**Change any value and re-run the cell to see what the model says.**

In [ ]:
# ── USER INPUT — edit any value below ─────────────────────────────────────────

custom_features = {

    # ── DA load forecasts (MW) — higher in summer, lower in shoulder ──────────
    # Typical summer peak:  CAPITL~1700, CENTRL~2200, LONGIL~3500, NYC~7500
    # Typical shoulder:     CAPITL~1200, CENTRL~1700, LONGIL~2000, NYC~5500
    'CAPITL_da_load_forecast': 1680.0,
    'CENTRL_da_load_forecast': 2180.0,
    'DUNWOD_da_load_forecast':  860.0,
    'GENESE_da_load_forecast': 1350.0,
    'HUDVL_da_load_forecast':  1420.0,
    'LONGIL_da_load_forecast': 3440.0,   # Long Island peaks in summer
    'MHKVL_da_load_forecast':  1180.0,
    'MILLWD_da_load_forecast':  410.0,
    'NORTH_da_load_forecast':   530.0,
    'NYC_da_load_forecast':    7820.0,   # NYC is largest demand zone
    'WEST_da_load_forecast':   2010.0,

    # ── DART lag-24h ($/MWh) — DART exactly 24 hours prior ───────────────────
    # Positive = yesterday DA > RT (DEC-favorable history)
    # Negative = yesterday RT > DA (INC-favorable history)
    # Normal range: -20 to +30. Extreme events: -1000 to +800.
    'CAPITL_dart_lag24':  12.5,
    'CENTRL_dart_lag24':  11.8,
    'DUNWOD_dart_lag24':  13.2,
    'GENESE_dart_lag24':  10.9,
    'HUDVL_dart_lag24':   12.1,
    'LONGIL_dart_lag24':  28.4,    # <-- try making this large e.g. 150
    'MHKVL_dart_lag24':   12.7,
    'MILLWD_dart_lag24':  13.0,
    'NORTH_dart_lag24':   11.5,
    'NYC_dart_lag24':      9.3,
    'WEST_dart_lag24':    10.8,

    # ── DART lag-48h ($/MWh) — DART exactly 48 hours prior ───────────────────
    'CAPITL_dart_lag48':   8.2,
    'CENTRL_dart_lag48':   7.6,
    'DUNWOD_dart_lag48':   9.1,
    'GENESE_dart_lag48':   7.0,
    'HUDVL_dart_lag48':    8.5,
    'LONGIL_dart_lag48':  18.3,
    'MHKVL_dart_lag48':    8.8,
    'MILLWD_dart_lag48':   9.0,
    'NORTH_dart_lag48':    7.4,
    'NYC_dart_lag48':       6.1,
    'WEST_dart_lag48':      7.2,

    # ── Load forecast error lag-24h (MW) — actual minus forecast, 24h ago ────
    # Positive = load was under-forecast (actual > forecast)
    # Negative = load was over-forecast (actual < forecast)
    # Typical range: -200 to +200 MW
    'CAPITL_lfe_lag24':   35.0,
    'CENTRL_lfe_lag24':   28.0,
    'DUNWOD_lfe_lag24':   12.0,
    'GENESE_lfe_lag24':   22.0,
    'HUDVL_lfe_lag24':    18.0,
    'LONGIL_lfe_lag24':   85.0,    # LONGIL was under-forecast yesterday
    'MHKVL_lfe_lag24':    15.0,
    'MILLWD_lfe_lag24':    8.0,
    'NORTH_lfe_lag24':    10.0,
    'NYC_lfe_lag24':      220.0,   # NYC was significantly under-forecast
    'WEST_lfe_lag24':      40.0,

    # ── Calendar features ─────────────────────────────────────────────────────
    'hour_of_day':    18,    # 18 = 6 PM, peak demand hour
    'month_of_year':   7,    # July
    'is_winter':       0,    # not winter
    'is_summer':       1,    # summer
    'is_weekend':      0,    # weekday
    'is_holiday':      0,    # not a holiday
}

# ──────────────────────────────────────────────────────────────────────────────

print('Running model on custom feature vector...')
print(f'Scenario: hour_of_day={custom_features["hour_of_day"]},',
      f'month={custom_features["month_of_year"]},',
      f'LONGIL_dart_lag24={custom_features["LONGIL_dart_lag24"]}')
print()
custom_results = predict(custom_features)

---
## Sensitivity analysis — what happens when you change one feature?

Take any base feature vector (from Mode 1 or Mode 2) and sweep one feature across a range.  
Useful for understanding what drives the model's decisions.

In [ ]:
# ── USER INPUT ─────────────────────────────────────────────────────────────────
BASE_FEATURES  = custom_features    # use the custom dict above, or replace with `features` from Mode 1
SWEEP_FEATURE  = 'LONGIL_dart_lag24'  # which feature to vary
SWEEP_VALUES   = [-50, -20, -10, 0, 5, 10, 20, 30, 50, 100, 200, 400]  # values to test
FOCUS_ZONE     = 'LONGIL'           # which zone to watch closely
# ──────────────────────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt

sweep_results = []
for val in SWEEP_VALUES:
    f = BASE_FEATURES.copy()
    f[SWEEP_FEATURE] = val
    r = predict(f, verbose=False)
    for _, row in r.iterrows():
        sweep_results.append({'value': val, 'zone': row.zone, 'side': row.side,
                              'probability': row.probability, 'fires': row.trade_fires})

sweep_df = pd.DataFrame(sweep_results)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: probability curves for FOCUS_ZONE both sides + threshold lines
ax = axes[0]
for side, color in [('pos', '#2e7d32'), ('neg', '#c62828')]:
    sub = sweep_df[(sweep_df.zone == FOCUS_ZONE) & (sweep_df.side == side)]
    ax.plot(sub['value'], sub['probability'], lw=2, color=color,
            label=f'{FOCUS_ZONE} {side.upper()} (DEC)' if side=='pos' else f'{FOCUS_ZONE} {side.upper()} (INC)')
    thr = THR.loc[(FOCUS_ZONE, side), 'best_tau']
    if pd.notna(thr) and THR.loc[(FOCUS_ZONE, side), 'eligible']:
        ax.axhline(thr, color=color, lw=0.8, ls='--', alpha=0.7)

ax.set_xlabel(f'{SWEEP_FEATURE} value ($/MWh)')
ax.set_ylabel('Predicted probability')
ax.set_title(f'Sensitivity: {FOCUS_ZONE} probability\nas {SWEEP_FEATURE} varies')
ax.legend()
ax.set_ylim(0, 1)
ax.axhline(0.5, color='grey', lw=0.5, ls=':')

# Right: number of trades fired across all zones
ax = axes[1]
n_fires = sweep_df.groupby('value')['fires'].sum()
ax.bar(range(len(SWEEP_VALUES)), n_fires.values, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(SWEEP_VALUES)))
ax.set_xticklabels([str(v) for v in SWEEP_VALUES], rotation=45)
ax.set_xlabel(f'{SWEEP_FEATURE} value')
ax.set_ylabel('Total trades fired (all zones)')
ax.set_title('Trades fired as feature varies')

plt.suptitle(f'Sensitivity analysis: sweeping {SWEEP_FEATURE}', y=1.02)
plt.tight_layout()
plt.show()

print(f'\nSummary table — {FOCUS_ZONE} probabilities:')
focus = sweep_df[sweep_df.zone == FOCUS_ZONE].pivot(index='value', columns='side', values='probability')
focus.columns = ['INC prob (neg)', 'DEC prob (pos)']
focus['DEC fires'] = focus['DEC prob (pos)'] >= float(THR.loc[(FOCUS_ZONE,'pos'),'best_tau'])
focus['INC fires'] = focus['INC prob (neg)'] >= float(THR.loc[(FOCUS_ZONE,'neg'),'best_tau'] or 1.0)
print(focus.to_string(float_format=lambda x: f'{x:.3f}'))

---
## Feature reference — what values to enter

### Zone-level features

| Feature | Unit | Typical range | Meaning |
|---|---|---|---|
| `{Z}_da_load_forecast` | MW | 250–8,000 | NYISO's published day-ahead forecast for the operating hour. Higher = more system stress. |
| `{Z}_dart_lag24` | \$/MWh | −500 to +600 | Realized DART exactly 24 hours before the operating hour. Positive = DA was above RT yesterday at this same time. |
| `{Z}_dart_lag48` | \$/MWh | −500 to +600 | Same, 48 hours back. Captures 2-day autocorrelation. |
| `{Z}_lfe_lag24` | MW | −500 to +500 | (actual load − forecast) at the corresponding hour yesterday. Positive = demand was under-forecasted. |

### Typical DA load forecasts by zone and season

| Zone | Summer peak | Winter peak | Shoulder |
|---|---:|---:|---:|
| CAPITL | 1,700 | 1,500 | 1,200 |
| CENTRL | 2,100 | 2,000 | 1,700 |
| DUNWOD | 950 | 620 | 600 |
| GENESE | 1,400 | 1,050 | 1,000 |
| HUDVL | 1,400 | 1,050 | 950 |
| LONGIL | 3,500 | 2,100 | 2,000 |
| MHKVL | 900 | 800 | 700 |
| MILLWD | 420 | 320 | 280 |
| NORTH | 540 | 610 | 500 |
| NYC | 8,000 | 5,200 | 5,500 |
| WEST | 2,050 | 1,650 | 1,550 |

### Calendar features

| Feature | Values | Notes |
|---|---|---|
| `hour_of_day` | 0–23 | 0 = midnight–1am, 18 = 6pm–7pm |
| `month_of_year` | 1–12 | 1=Jan … 12=Dec |
| `is_winter` | 0 or 1 | 1 if month is 12, 1, or 2 |
| `is_summer` | 0 or 1 | 1 if month is 6, 7, or 8 |
| `is_weekend` | 0 or 1 | 1 if Saturday or Sunday |
| `is_holiday` | 0 or 1 | 1 if US federal holiday (New Year, July 4, Thanksgiving, etc.) |

### Quick scenarios to try

**Extreme winter cold snap** — copy the Jan 28 2026 pattern:
- Set `dart_lag24` for all zones to ~500
- Set `dart_lag48` for all zones to ~280
- Set `da_load_forecast` to 1.3× the summer peak values
- Set `hour_of_day=7`, `month_of_year=1`, `is_winter=1`
- Expect: almost all DEC trades fire at very high probability

**Quiet shoulder day** — copy the Apr 10 2025 pattern:
- Set `dart_lag24` for all zones to 10–20
- Set `da_load_forecast` to shoulder values above
- Set `hour_of_day=12`, `month_of_year=4`, `is_winter=0`, `is_summer=0`
- Expect: a few DEC trades fire just above τ, most sit out

**LONGIL INC signal** — set up conditions for a negative DART spike:
- Set `LONGIL_dart_lag24` to −80 (yesterday RT was above DA)
- Set `LONGIL_da_load_forecast` to 3600
- Set `hour_of_day=19`, `month_of_year=8`, `is_summer=1`
- Expect: LONGIL INC (neg side) probability increases

In [ ]:
# ── Quick helper: print a blank feature template you can copy-paste ───────────
print('BLANK FEATURE TEMPLATE — copy this into Mode 2 and fill in values:')
print()
print('custom_features = {')
for f in ZONE_FEATS:
    print(f'    # {f}')
    for z in ZONES:
        print(f"    '{z}_{f}': 0.0,")
    print()
print('    # Calendar')
for c in CAL_FEATS:
    print(f"    '{c}': 0,")
print('}')

In [ ]:
# ── Check model internals for any zone/side ──────────────────────────────────
# Change these to inspect a different model
INSPECT_ZONE = 'LONGIL'
INSPECT_SIDE = 'neg'

label  = f'{INSPECT_ZONE}_{INSPECT_SIDE}'
pipe   = PIPES[label]
scaler = pipe.named_steps['scaler']
logreg = pipe.named_steps['logreg']

coef_df = pd.DataFrame({
    'feature':      FEATURE_COLS,
    'train_mean':   scaler.mean_,
    'train_std':    scaler.scale_,
    'coefficient':  logreg.coef_[0],
}).sort_values('coefficient', key=abs, ascending=False)

thr_row = THR.loc[(INSPECT_ZONE, INSPECT_SIDE)]
print(f'Model: {label}')
print(f'Intercept:  {logreg.intercept_[0]:+.4f}')
print(f'Threshold:  {thr_row["best_tau"]}')
print(f'Eligible:   {thr_row["eligible"]}')
print(f'Val P&L:    ${thr_row["val_pnl"]:,.2f}')
print()
print('Top 15 features by absolute coefficient:')
print(coef_df.head(15)[['feature','coefficient','train_mean','train_std']]
      .to_string(index=False, float_format=lambda x: f'{x:+.4f}'))